# Topic merging

This notebook:

- Merges two BERTopic models created during an initial run and a subsequent outlier run
- Calculates corrected document counts for each models and merged topic assignments
- Annotates the data with human-generated topic labels
- Links back topics to documents & videos

_____________
# Importing packages

In [1]:
import pickle
import numpy as np
import pandas as pd

from config import *    # Import config for paths
from pathlib import Path
from collections import Counter
from bertopic import BERTopic

# Loading data and topic models

In [2]:
# Load original data
hashtag_df = pd.read_pickle(HASHTAG_DF_PATH)

In [3]:
# Reloading all documents
with open(FINAL_DOCS_PATH, "rb") as f:
    docs_bert = pickle.load(f)

# Reloading outlier documents
with open(OUTLIER_DOCS_PATH, "rb") as f:
    outlier_docs = pickle.load(f)

# Reload topic model
topic_model = BERTopic.load(TOPIC_MODEL_PATH)
topics = np.load(TOPICS_ASSIGNED_PATH)

# Reload outlier model
topic_model_outliers = BERTopic.load(OUTLIER_MODEL_PATH)
topics_outliers = np.load(OUTLIER_TOPICS_PATH)

In [4]:
# Verify data
print(f"Nr. of videos: {len(hashtag_df)}")
print(f"Nr. of documents: {len(docs_bert)}")
print(f"Nr. of main model topics: {len(topics)}")
print(f"\nNr. of outlier documents: {len(outlier_docs)}")
print(f"Nr. of outlier topics: {len(topics_outliers)}")

Nr. of videos: 5561559
Nr. of documents: 5332704
Nr. of main model topics: 5332704

Nr. of outlier documents: 1541249
Nr. of outlier topics: 1541249


In [7]:
hashtag_df.tail(2)

,hashtags,hashtags_parented
5561557,"[repost, explore, fyp, foryoupagе, foryou, aut...","[fall, cold, relatable, fyp, imold, autumn, re..."
5561558,"[paris, eiffeltower, toureiffel, fyp, solotrav...","[fyp, paris, toureiffel, solotravel, eiffeltow..."


# Corrected topic counts

The method `.get_topic_info()` will not show correct counts of the number of documents in each topic due to the batch processing on batches of *n* = 50,000 used during topic modelling in `2_topic_modelling.ipynb`.

The code below re-calculates counts for the `.get_topic_info()` method for the original run topics and outlier run topics and exports them as readable excel sheets.

In [5]:
def get_corrected_topic_info(topic_model, topics):
    """
    Args:
        topic_model: BERTopic model (trained on subset)
        topics: Full topic assignments array (all 5.3M documents)
    
    Returns:
        DataFrame with corrected counts
    """
    # Get base info from model
    base_info = topic_model.get_topic_info()
    
    # Calculate TRUE counts from actual assignments
    true_counts = Counter(topics)
    
    # Update the Count column with true values
    base_info['Count'] = base_info['Topic'].map(true_counts).fillna(0).astype(int)
    
    # Re-sort by count (descending)
    corrected_info = base_info.sort_values('Count', ascending=False).reset_index(drop=True)
    
    return corrected_info

Corrected topic info for the main model.

In [6]:
# Apply to our main model
corrected_topic_info = get_corrected_topic_info(topic_model, topics)

print("\n" + "="*70)
print("CORRECTED MAIN MODEL TOPIC COUNTS")
print("="*70)
print(f"Total documents: {corrected_topic_info['Count'].sum():,}")
print(f"Number of topics: {len(corrected_topic_info[corrected_topic_info['Topic'] != -1])}")

# Show comparison
print(f"\n📊 Count Verification:")
print(f"  Model reports (WRONG): {topic_model.get_topic_info()['Count'].sum():,}")
print(f"  Actual assignments: {len(topics):,}")
print(f"  Corrected counts: {corrected_topic_info['Count'].sum():,}")
print(f"  Match: {'✅' if corrected_topic_info['Count'].sum() == len(topics) else '❌'}")


CORRECTED MAIN MODEL TOPIC COUNTS
Total documents: 5,332,704
Number of topics: 228

📊 Count Verification:
  Model reports (WRONG): 50,000
  Actual assignments: 5,332,704
  Corrected counts: 5,332,704
  Match: ✅


In [12]:
corrected_topic_info.head(2)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1541249,-1_meme_anime_funny_bnha,"[meme, anime, funny, bnha, mha, greenscreen, c...","[yougetmeyuh ehenn carbon hayeckpuur, memes😂 😂..."
1,6,154565,6_the100_julieandthephantoms_charlidamelio_reign,"[the100, julieandthephantoms, charlidamelio, r...",[reign queenvictoria maryqueenotscots reignedi...


In [ ]:
# Save file
corrected_topic_info.to_excel(CORRECTED_TOPIC_INFO_PATH, index=False)

Corrected topic info for the outlier model.

In [7]:
# Correct the counts for the outlier
corrected_outlier_info = get_corrected_topic_info(topic_model_outliers, topics_outliers)

print("\n" + "="*70)
print("CORRECTED OUTLIER MODEL TOPIC COUNTS")
print("="*70)
print(f"Total outlier documents processed: {corrected_outlier_info['Count'].sum():,}")
print(f"Number of niche topics: {len(corrected_outlier_info[corrected_outlier_info['Topic'] != -1])}")

# Show comparison
print(f"\n📊 Count Verification:")
print(f"  Model reports (WRONG): {topic_model_outliers.get_topic_info()['Count'].sum():,}")
print(f"  Actual assignments: {len(topics_outliers):,}")
print(f"  Corrected counts: {corrected_outlier_info['Count'].sum():,}")
print(f"  Match: {'✅' if corrected_outlier_info['Count'].sum() == len(topics_outliers) else '❌'}")


CORRECTED OUTLIER MODEL TOPIC COUNTS
Total outlier documents processed: 1,541,249
Number of niche topics: 248

📊 Count Verification:
  Model reports (WRONG): 50,000
  Actual assignments: 1,541,249
  Corrected counts: 1,541,249
  Match: ✅


In [13]:
corrected_outlier_info.head(2)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,569288,-1_music_cute_fashion_funny,"[music, cute, fashion, funny, anime, love, rel...",[alttiktok stitch kuromi cosplay cute hellokit...
1,1,27308,1_taylorswift_swifttok_erastour_swiftie,"[taylorswift, swifttok, erastour, swiftie, tay...",[wizzair erastour swiftie taylorswift swifttok...


In [ ]:
# Save file
corrected_outlier_info.to_excel(CORRECTED_OUTLIER_INFO_PATH, index=False)

# Merging topics

Below the two models are merged via their topic assignments.

In [8]:
# Verification of match
print("VERIFICATION BEFORE MERGING")
print("-"*70)

outliers_in_main = (topics == -1).sum()
docs_in_outlier_model = len(topics_outliers)

print(f"Main model:")
print(f"  Total docs: {len(topics):,}")
print(f"  Outliers: {outliers_in_main:,}")
print(f"  Assigned: {(topics != -1).sum():,}")
print(f"\nOutlier model:")
print(f"  Total docs: {len(topics_outliers):,}")
print(f"  Still outliers: {(topics_outliers == -1).sum():,}")
print(f"  Assigned: {(topics_outliers != -1).sum():,}")
print(f"\n✅ Match: {outliers_in_main == docs_in_outlier_model}")

if outliers_in_main != docs_in_outlier_model:
    print("⚠️  WARNING: Counts don't match - check if models belong together")

VERIFICATION BEFORE MERGING
----------------------------------------------------------------------
Main model:
  Total docs: 5,332,704
  Outliers: 1,541,249
  Assigned: 3,791,455

Outlier model:
  Total docs: 1,541,249
  Still outliers: 569,288
  Assigned: 971,961

✅ Match: True


`merge_topic_assignments`

This function:

- Finds the maximum topic ID from the main topic model.
- Adds an offset so outlier model topics won't conflict.

All -1 topics in the main model output are replaced with the correctly offset outlier topics.

Returns a clean, merged topic assignment array and a mapping of original → merged topic IDs.

In [9]:
def merge_topic_assignments(topics_main, topics_outliers, topic_model_main, topic_model_outliers):
    """    
    Args:
        topics_main: Topic assignments from main model (all 5.3M docs)
        topics_outliers: Topic assignments from outlier model (outlier docs only)
        topic_model_main: Main BERTopic model
        topic_model_outliers: Outlier BERTopic model
        
    Returns:
        merged_topics: Combined topic assignments
        topic_id_mapping: Dictionary mapping original IDs to merged IDs
    """
    print("MERGING TOPIC ASSIGNMENTS")
    print("-"*70)
    
    # Find the offset for outlier topics
    max_main_topic = topics_main[topics_main != -1].max()
    offset = max_main_topic + 1
    
    print(f"\nMain model topic range: 0 to {max_main_topic}")
    print(f"Offset for outlier topics: {offset}")
    
    # Create merged array (start with main model topics)
    merged_topics = topics_main.copy()
    
    # Offset outlier model topic IDs (but keep -1 as -1)
    outlier_topics_offset = topics_outliers.copy()
    non_outlier_mask = outlier_topics_offset != -1
    outlier_topics_offset[non_outlier_mask] += offset
    
    max_outlier_topic = outlier_topics_offset[non_outlier_mask].max() if non_outlier_mask.any() else offset
    print(f"Outlier model topic range after offset: {offset} to {max_outlier_topic}")
    
    # Get indices of outliers in main model
    outlier_indices = np.where(topics_main == -1)[0]
    print(f"\nReplacing {len(outlier_indices):,} outliers with new assignments...")
    
    # Verify lengths match
    if len(outlier_indices) != len(outlier_topics_offset):
        print(f"❌ ERROR: Length mismatch!")
        print(f"   Expected {len(outlier_indices):,} but got {len(outlier_topics_offset):,}")
        return None, None
    
    # Replace outliers with offset topics
    merged_topics[outlier_indices] = outlier_topics_offset
    
    # Create mapping for reference
    topic_id_mapping = {
        'main_topics': list(range(0, max_main_topic + 1)),
        'outlier_topics_original': list(range(0, max_outlier_topic - offset + 1)),
        'outlier_topics_merged': list(range(offset, max_outlier_topic + 1)),
        'offset': offset
    }
    
    # Summary
    print("\n" + "*"*70)
    print("\nMERGE SUMMARY")
    print("-"*70)
    print(f"Total documents: {len(merged_topics):,}")
    print(f"Unique topics (from main): {len(np.unique(topics_main[topics_main != -1]))}")
    print(f"Unique topics (from outlier): {len(np.unique(topics_outliers[topics_outliers != -1]))}")
    print(f"Total unique topics (merged): {len(np.unique(merged_topics[merged_topics != -1]))}")
    print(f"Final outliers: {(merged_topics == -1).sum():,} ({(merged_topics == -1).sum()/len(merged_topics)*100:.2f}%)")
    
    return merged_topics, topic_id_mapping

In [10]:
# Execute merge
merged_topics, topic_id_mapping = merge_topic_assignments(
    topics_main=topics,
    topics_outliers=topics_outliers,
    topic_model_main=topic_model,
    topic_model_outliers=topic_model_outliers
)

MERGING TOPIC ASSIGNMENTS
----------------------------------------------------------------------

Main model topic range: 0 to 227
Offset for outlier topics: 228
Outlier model topic range after offset: 228 to 475

Replacing 1,541,249 outliers with new assignments...

**********************************************************************

MERGE SUMMARY
----------------------------------------------------------------------
Total documents: 5,332,704
Unique topics (from main): 228
Unique topics (from outlier): 248
Total unique topics (merged): 476
Final outliers: 569,288 (10.68%)


In [ ]:
# Save merged topics
np.save(MERGED_TOPICS_PATH, merged_topics)

# Corrected merged counts

`merge_topic_info_corrected` 

This function creates a single, merged, accurate “topic_info” table after merging:

- Main BERTopic model topic info
- Outlier BERTopic model topic info
- The merged assignment array (merged_topics)

In [11]:
def merge_topic_info_corrected(corrected_main_info, corrected_outlier_info, 
                                          merged_topics, offset):
    """
    Args:
        corrected_main_info: Topic info from main model with corrected counts
        corrected_outlier_info: Topic info from outlier model corrected counts
        merged_topics: The actual merged topic assignments array created above
        offset: Offset value for outlier topic IDs
        
    Returns:
        merged_topic_info: Combined DataFrame with TRUE counts
    """
    
    print("CREATING MERGED TOPIC INFO TABLE")
    print("-"*70)
    
    # Calculate counts from merged_topics
    true_counts = Counter(merged_topics)
    
    print(f"\nCounts from merged_topics:")
    print(f"  Total documents: {len(merged_topics):,}")
    print(f"  Outliers (-1): {true_counts[-1]:,}")
    print(f"  Assigned to topics: {len(merged_topics) - true_counts[-1]:,}")
    
    # Copy DataFrames
    main_info = corrected_main_info.copy()
    outlier_info = corrected_outlier_info.copy()
    
    # Offset outlier topic IDs (except -1)
    outlier_info.loc[outlier_info['Topic'] != -1, 'Topic'] += offset
    
    # Remove outlier row from outlier_info (will use main model's outlier row)
    outlier_info = outlier_info[outlier_info['Topic'] != -1]
    
    # Add source column
    if 'Source' not in main_info.columns:
        main_info['Source'] = 'Main Model'
    if 'Source' not in outlier_info.columns:
        outlier_info['Source'] = 'Outlier Model'
    
    # Combine
    combined_info = pd.concat([main_info, outlier_info], ignore_index=True)
    
    # Update counts with true values from merged_topics
    combined_info['Count'] = combined_info['Topic'].map(true_counts).fillna(0).astype(int)
    
    # Remove topics with 0 count (topics that don't appear in merged_topics)
    combined_info = combined_info[combined_info['Count'] > 0]
    
    # Sort outliers first, then by count descending
    outliers = combined_info[combined_info['Topic'] == -1]
    non_outliers = combined_info[combined_info['Topic'] != -1].sort_values('Count', ascending=False)
    
    merged_info = pd.concat([outliers, non_outliers]).reset_index(drop=True)
    
    # Print summary
    print(f"\n Merged topic info created with correct counts")
    print(f"   Total rows: {len(merged_info)}")
    print(f"   Total documents: {merged_info['Count'].sum():,}")
    print(f"   Outliers: {merged_info[merged_info['Topic'] == -1]['Count'].values[0]:,}")
    
    return merged_info

In [12]:
# Apply the function
merged_topic_info = merge_topic_info_corrected(
    corrected_main_info=corrected_topic_info,
    corrected_outlier_info=corrected_outlier_info,
    merged_topics=merged_topics,
    offset=topic_id_mapping['offset']
)

CREATING MERGED TOPIC INFO TABLE
----------------------------------------------------------------------

Counts from merged_topics:
  Total documents: 5,332,704
  Outliers (-1): 569,288
  Assigned to topics: 4,763,416

 Merged topic info created with correct counts
   Total rows: 477
   Total documents: 5,332,704
   Outliers: 569,288


In [13]:
print(len(merged_topic_info))
merged_topic_info.sample(3, random_state=42)

477


,Topic,Count,Name,Representation,Representative_Docs,Source
468,430,1072,202_psoriasis_endometriosis_adenomyosis_womens...,"[psoriasis, endometriosis, adenomyosis, womens...",[eczematok psoriasis tsw eczemaawareness eczem...,Outlier Model
33,13,33839,13_christmas_happyholidays_gift_monclerbubbleup,"[christmas, happyholidays, gift, monclerbubble...","[christmas secretsanta, christmas santaslays, ...",Main Model
131,53,8676,53_greenscreenvideo_greenscreen_snapchat_googl...,"[greenscreenvideo, greenscreen, snapchat, goog...","[greenscreenvideo, greenscreenvideo, greenscre...",Main Model


# Renaming topic labels based on human coding

Topics were named during qualitative coding sessions, identifying labels, duplicates/overlapping, or incoherent topics.

`rename_merged_topic_labels`

This function takes:

- The merged topic info table
- Dictionaries of new labels for both models
- The offset used during merging

…and returns the same table but with all topic names updated to the new human-readable labels.

In [13]:
def rename_merged_topic_labels(merged_topic_info, main_topic_mapping, outlier_topic_mapping, offset):
    """
    Args:
        merged_topic_info: Merged topic info DataFrame
        main_topic_mapping: Dict mapping main model topic_id -> new_label
        outlier_topic_mapping: Dict mapping outlier model topic_id -> new_label
        offset: Offset used for outlier topics
        
    Returns:
        Updated DataFrame with new labels
    """
    df = merged_topic_info.copy()
    
    # Keep original labels for reference
    df['Original Topic Label'] = df['Name']
    
    # Create combined mapping (offset outlier topic IDs)
    combined_mapping = {}
    
    # Add main model mappings
    combined_mapping.update(main_topic_mapping)
    
    # Add outlier model mappings (with offset)
    for orig_id, new_label in outlier_topic_mapping.items():
        if orig_id != -1:  # Don't offset outliers
            offset_id = orig_id + offset
            combined_mapping[offset_id] = new_label
    
    # Apply the mapping
    df['Name'] = df['Topic'].map(combined_mapping).fillna(df['Name'])
    
    # Count how many were renamed
    renamed_count = (df['Name'] != df['Original Topic Label']).sum()
    
    print(f"✅ Renamed {renamed_count} topic labels")

    # Rename 'Name' column
    df = df.rename(columns={'Name': 'Topic Label'})
    
    return df

Below two dictionaries are created where the IDs need to correspond to the original topic IDs from the main model and outlier model, found in:

manual coding/Final topics IDs.xlsx  
manual coding/Incoherent topics IDs.xlsx             

In [14]:
# Main model topic labels
main_topic_mapping = {
    -1: 'Outliers', 0: 'Cooking', 1: 'Art', 2: 'Incoherent', 3: 'Dogs',
    4: 'Harry Potter', 5: 'LGBTQ+', 6: 'TV shows/movies', 7: 'Fitness',
    8: 'Hair', 9: 'School', 10: 'Fashion', 11: 'Netherlands', 12: 'Halloween',
    13: 'Christmas', 14: 'Gaming', 15: 'Funny', 16: 'Disney', 17: 'TV shows/movies',
    18: 'Make-up', 19: 'Incoherent', 20: 'Incoherent', 21: 'Incoherent', 22: 'Marvel',
    23: 'Incoherent', 24: 'Couples', 25: 'Animal Crossing', 26: 'Parenting',
    27: 'Dance', 28: 'Incoherent', 29: 'Anime', 30: 'Football', 31: 'K-Pop',
    32: 'Cats', 33: 'My Hero Academia', 34: 'Witchcraft', 35: 'Skincare', 36: 'COVID-19',
    37: 'Gymnastics', 38: 'The Vampire Diaries', 39: 'Asia', 40: 'Furries',
    41: 'Cars', 42: 'Goth', 43: 'Naruto', 44: 'Avatar: The Last Airbender', 
    45: 'The Boyz (Kpop)', 46: 'Friends', 47: 'Basketball', 48: 'My Hero Academia',
    49: 'Dangan Ronpa', 50: 'Photography', 51: 'iPhone', 52: 'Education', 53: 'Incoherent',
    54: 'Fish', 55: 'Incoherent', 56: 'Incoherent', 57: 'Small business', 58: 'Incoherent',
    59: 'Summer', 60: 'Books', 61: 'Singing', 62: 'Incoherent', 63: 'Cosplay',
    64: 'Horses', 65: 'My Hero Academia', 66: 'Funny', 67: 'Storytime', 68: 'Beach', 69: 'Challenges',
    70: 'Animation', 71: 'Gifts', 72: 'Astrology', 73: 'Couples Pranks', 74: 'Quarantine',
    75: 'Languages', 76: 'Europe', 77: 'Incoherent', 78: 'Incoherent', 79: 'Mental Health',
    80: 'Dating', 81: 'Siblings',  82: 'Incoherent', 83: 'Motorcycles', 84: 'NCT', 85: 'Incoherent', 
    86: 'Acting', 87: 'Funny', 88: 'Ski/Snowboard', 89: 'Sewing', 90: 'Travel',
    91: 'Commonwealth', 92: 'Sneakers/Shoes', 93: 'Interior Design', 94: 'Incoherent',
    95: 'Guitar', 96: 'Stray Kidz', 97: 'Black Lives Matter', 98: 'Among Us', 99: 'Musical Theatre',
    100: 'Journalling', 101: 'My Hero Academia', 102: 'Skateboarding', 103: 'Tutorials/How to',
    104: 'Mystic Messenger', 105: 'Roblox', 106: 'Bugha', 107: 'My Hero Academia',
    108: 'My Hero Academia', 109: 'Fortnite', 110: 'Military', 111: 'Incoherent',
    112: 'Pranks', 113: 'Snakes', 114: 'Magic Tricks', 115: 'Incoherent', 116: 'Coffee',
    117: 'Music production', 118: 'Minecraft', 119: 'Incoherent', 120: 'Nails', 121: 'Incoherent',
    122: 'Police', 123: 'Hip Hop/Rap', 124: 'Swimming', 125: 'Body modification',
    126: 'Instagram', 127: 'Birds', 128: 'Sad quotes', 129: 'Memes', 130: 'Snapchat',
    131: 'Funny', 132: 'Celebrations/Events', 133: 'Tom Holland & Zendaya', 134: 'Voice Effects',
    135: 'Incoherent', 136: 'Incoherent', 137: 'Incoherent', 138: 'Chiropractor', 139: 'Sad/depressed',
    140: 'Minecraft Streamers', 141: 'Incoherent', 142: 'Animal Crossing', 143: 'Vibes', 144: 'Rocket League',
    145: 'K-Pop', 146: 'Cleaning', 147: 'Life Hacks', 148: 'Splatoon', 149: 'Incoherent', 150: 'Incoherent',
    151: 'Gossip Girl', 152: 'Twins', 153: 'Animals', 154: 'Funny', 155: 'Incoherent', 156: 'Autism',
    157: 'Piano', 158: 'Incoherent', 159: 'Incoherent', 160: 'Incoherent', 161: 'Singing',
    162: 'Star Wars: Battlefront II', 163: 'Metal Music', 164: 'Farming', 165: 'Weddings',
    166: 'Arab/Muslim', 167: 'Tourettes', 168: 'Hazbin Hotel', 169: 'One Direction', 170: 'Girls',
    171: 'Tattoos', 172: 'Greys Anatomy', 173: 'Memories', 174: 'Thai Boys Love shows',
    175: 'Toilet-Bound Hanako-kun', 176: 'Incoherent', 177: 'Football', 178: 'Violin',
    179: 'One Direction', 180: 'Incoherent', 181: 'Attack on Titan', 182: 'Incoherent', 
    183: 'Percy Jackson', 184: 'Incoherent', 185: 'Friends (TV Show)', 186: 'New Year',
    187: 'Minecraft', 188: 'EDM', 189: 'Sleep', 190: 'Riverdale', 191: 'The Chronicles of Narnia',
    192: 'Cats', 193: 'Psychology', 194: 'Funny', 195: 'Glow-up', 196: 'My Hero Academia',
    197: 'My Hero Academia', 198: 'Christianity', 199: 'My Hero Academia', 200: 'Trucking',
    201: 'FIFA/EA', 202: 'My Hero Academia', 203: 'Belgium', 204: 'Insects', 205: 'Amazon',
    206: 'Spotify', 207: 'Vlogs', 208: 'Incoherent', 209: 'Fairy Tale', 210: 'Alcohol',
    211: 'Bridgerton', 212: 'Incoherent', 213: 'Criminal Minds', 214: 'Ateez', 215: 'Incoherent',
    216: 'Hamsters', 217: 'Teeth', 218: 'Plumbing', 219: 'Incoherent', 220: 'Snow/Ice/Winter',
    221: 'My Hero Academia', 222: 'Dance', 223: 'Dinosaurs', 224: '2024 US election', 225: 'Food', 226: 'Pregnancy',
    227: 'Thrifting'

}

In [15]:
# Outlier model topic labels
outlier_topic_mapping = {
    -1: 'Outliers', 0: 'Football', 1: 'Taylor Swift', 2: 'Funny', 3: 'Genshin Impact', 4: 'Incoherent',
    5: 'K-Pop', 6: 'Motivational', 7: 'Cars', 8: 'Cats', 9: 'Incoherent', 10: 'Martial arts', 11: 'Eurovision',
    12: 'Bankzitters', 13: 'Game of Thrones', 14: 'Netherlands', 15: 'ASMR', 
    16: 'Couples', 17: 'Karate Kid', 18: 'Dance', 19: 'The Hunger Games', 20: 'Feminism', 21: 'Billie Eilish', 22: 'Jujutsu Kaisen',
    23: 'Minecraft', 24: 'Dark Romance', 25: 'Making Money', 26: 'Formula 1', 27: 'Hockey', 28: 'Incoherent',
    29: 'Ateez', 30: 'Incoherent', 31: 'CapCut Editing', 32: 'Incoherent', 33: 'Geography', 34: 'Sped up music',
    35: 'Podcasts/radio', 36: 'Cosplay', 37: 'Manhwa', 38: 'Work', 39: 'Facts', 40: 'Pregnancy', 41: 'Football',
    42: 'Music', 43: 'Cooking', 44: 'Perfume', 45: 'Lego', 46: 'Hatsune Miku', 47: 'The Walking Dead', 48: 'Theme Parks',
    49: 'Poetry', 50: 'Advice', 51: 'Eating disorder recovery', 52: 'Scandinavia', 53: 'Incoherent', 54: 'Incoherent',
    55: 'Incoherent', 56: 'Brawl Stars', 57: 'Wealth', 58: 'Africa', 59: 'Twitch streamers', 60: 'Snow/Ice/Winter',
    61: 'Incoherent', 62: 'Grief', 63: 'Funny', 64: 'Music festivals', 65: 'Friends', 66: 'Tomorrow x Together',
    67: 'Demon Slayer', 68: 'France', 69: 'Positivity', 70: 'Fortnite', 71: 'Shopping', 72: 'Couples', 73: 'Gaming',
    74: 'The Sims', 75: 'Titanic (movie)', 76: 'Family Guy', 77: 'Incoherent', 78: 'Incoherent', 79: 'Girls',
    80: 'The Weeknd', 81: 'Justin Bieber', 82: 'NBA 2k', 83: 'Incoherent', 84: 'Farming', 85: 'Turkey',
    86: 'Construction', 87: 'Incoherent', 88: 'NFL', 89: 'Rainbow Six Siege', 90: 'Twice', 91: 'Heartbreak',
    92: 'Motivational', 93: 'Steven Universe', 94: 'Pirates of the Caribbean', 95: 'Audio', 96: 'Driving lessons',
    97: 'Singing', 98: 'South Asia', 99: 'History', 100: 'Total Drama Island', 101: 'Trauma', 102: 'Fanfiction',
    103: 'Bridgerton', 104: 'Food', 105: 'Love Island', 106: 'Prom', 107: 'Once Upon A Time', 108: 'Rappers',
    109: 'Ariana Grande', 110: 'Reddit stories', 111: 'Music', 112: 'Plants/gardening', 113: 'Call of Duty',
    114: 'Fitness', 115: 'Clash Royale', 116: 'Sturniolo triplets', 117: 'Quotes', 118: 'Greece', 119: 'Street interviews',
    120: 'Generations', 121: 'Incoherent', 122: 'Resin art', 123: 'Clothing shopping', 124: 'Italy', 125: 'Incoherent',
    126: 'Incoherent', 127: 'Latin America/Caribbean', 128: 'Incoherent', 129: 'Balkans', 130: 'Concerts', 131: 'Blue Lock',
    132: 'Incoherent', 133: 'Sam and Colby', 134: 'Studying', 135: 'Miraculous Ladybug', 136: 'Modelling (fashion)',
    137: 'Reddit stories', 138: 'Incoherent', 139: 'Memes', 140: 'Rappers', 141: 'My Hero Academia', 142: 'Incoherent',
    143: 'Enhypen', 144: 'Animals', 145: 'Enhypen', 146: 'Incoherent', 147: 'Gidle', 148: 'The Amazing Digital Circus',
    149: 'Pinterest aesthetic', 150: 'Vintage fashion', 151: 'Vlogs', 152: 'Driving lessons', 153: 'Taylor Swift',
    154: 'Fashion', 155: 'Body positivity', 156: 'Spongebob', 157: 'K2 zoekt K3', 158: 'Incoherent', 159: 'Incoherent',
    160: 'Disability', 161: 'Fashion', 162: 'Rabbits', 163: 'Chronic illness/diseases', 164: 'Van life', 165: 'Incoherent',
    166: 'Harry Styles', 167: 'Golf', 168: 'Fire/rescue', 169: 'Incoherent', 170: 'Languages', 171: 'Incoherent',
    172: '3D printing', 173: 'Incoherent', 174: 'The Summer I Turned Pretty', 175: 'Incoherent', 176: 'Incoherent',
    177: 'Swimsuits', 178: 'Incoherent', 179: 'Bungou Stray Dogs', 180: 'Cristiano Ronaldo', 181: 'Incoherent', 182: 'Flowers',
    183: 'Art', 184: 'Fortnite', 185: 'Feyenoord FC', 186: 'Retro music', 187: 'Incoherent', 188: 'Lawyers', 189: 'Drums',
    190: 'Incoherent', 191: 'Incoherent', 192: 'Incoherent', 193: 'Couples', 194: 'Incoherent', 195: 'Rappers',
    196: 'My little pony', 197: 'Olivia Rodrigo', 198: 'Glee', 199: 'Incoherent', 200: 'Body positivity',
    201: 'Incoherent', 202: 'Chronic illness/diseases', 203: 'The Voice', 204: 'Heartbreak', 205: 'Incoherent',
    206: 'Heartstopper', 207: 'How to Train Your Dragon', 208: 'Incoherent', 209: 'Science', 210: 'Incoherent',
    211: 'Love', 212: 'Incoherent', 213: 'Incoherent', 214: 'Brugklas', 215: 'Incoherent', 216: 'Incoherent',
    217: 'Cancer', 218: 'Incoherent', 219: 'Sad/depressed', 220: 'Ice skating', 221: 'Books', 222: 'FIFA/EA',
    223: 'Old money', 224: 'Outer Banks', 225: 'Minecraft', 226: 'Incoherent', 227: 'Incoherent', 228: 'Self-love',
    229: 'Incoherent', 230: 'One Piece', 231: 'Toxic relationships', 232: 'Lord of the Rings', 233: 'Incoherent',
    234: 'Sidemen', 235: 'Europe', 236: 'Wednesday Addams', 237: 'Incoherent', 238: 'Incoherent', 239: 'Monster High',
    240: 'Incoherent', 241: 'Incoherent', 242: 'Backpacking', 243: 'Incoherent', 244: 'Incoherent', 245: 'Sustainability',
    246: 'Nails', 247: 'Incoherent'
}

In [16]:
# Run the function to rename labels
merged_topic_info_renamed = rename_merged_topic_labels(
    merged_topic_info=merged_topic_info,
    main_topic_mapping=main_topic_mapping,
    outlier_topic_mapping=outlier_topic_mapping,
    offset=topic_id_mapping['offset']
)

✅ Renamed 477 topic labels


In [18]:
merged_topic_info_renamed.sample(2, random_state=42)

,Topic,Count,Topic Label,Representation,Representative_Docs,Source,Original Topic Label
468,430,1072,Chronic illness/diseases,"[psoriasis, endometriosis, adenomyosis, womens...",[eczematok psoriasis tsw eczemaawareness eczem...,Outlier Model,202_psoriasis_endometriosis_adenomyosis_womens...
33,13,33839,Christmas,"[christmas, happyholidays, gift, monclerbubble...","[christmas secretsanta, christmas santaslays, ...",Main Model,13_christmas_happyholidays_gift_monclerbubbleup


# Adding overarching topics (categories)

With 476 initial topics, we deduced that some of them are closely related, e.g., referring to jobs, communities, media franchises etc. 

Those were manually labeled and put together, and the code below applies this step to be reflected in the data frame, i.e., adding an additional column indicated which parent topic each topic belongs to.

In [17]:
# Helper function to add categories to topics
def convert_category_mapping(reverse_mapping):
    """Convert category->topics to topic->category format"""
    converted = {'main': {}, 'outlier': {}}
    
    for model_type in ['main', 'outlier']:
        if model_type in reverse_mapping:
            for parent, topic_ids in reverse_mapping[model_type].items():
                for topic_id in topic_ids:
                    converted[model_type][topic_id] = parent
    
    return converted

`add_categories()`

This function adds a category column to the merged BERTopic topic info table.

Because we merged main-model topics and outlier-model topics (with an offset), ywe need a unified way to assign each topic to its overarching category (our human-curated grouping).

This function creates that final category assignment.

In [19]:
def add_categories(merged_topic_info, category_mapping, offset, mapping_format='topic_to_category'):
    """
    Add category column to merged_topic_info.
    
    Args:
        merged_topic_info: Merged topic info DataFrame
        category_mapping: Dict with 'main' and 'outlier' category mappings
        offset: Offset for outlier topics
        mapping_format: 'topic_to_category' (default) or 'category_to_topics'
        
    Returns:
        DataFrame with 'category' column added
    """
    df = merged_topic_info.copy()
    
    # Convert if mapping is in category -> topics format
    if mapping_format == 'category_to_topics':
        category_mapping = convert_category_mapping(category_mapping)  # still works, same structure
    
    # Create unified category mapping
    combined_category_mapping = {}
    
    # Main model category assignments
    if 'main' in category_mapping:
        combined_category_mapping.update(category_mapping['main'])
    
    # Outlier model category assignments (apply offset)
    if 'outlier' in category_mapping:
        for topic_id, category in category_mapping['outlier'].items():
            if topic_id != -1:
                offset_id = topic_id + offset
                combined_category_mapping[offset_id] = category
    
    # Apply the category mapping
    df['Category'] = df['Topic'].map(combined_category_mapping).fillna('Uncategorized')
    
    # Assign outliers
    df.loc[df['Topic'] == -1, 'Category'] = 'Outlier'
    
    print("✅ Added categories")
    
    return df


In [20]:
# Based on manual step
category_topic_mapping = {
    'main': {
        'Comedy': [15, 66, 87, 131, 154, 194, 73, 112, 129],
        'Food and drinks': [0, 116, 210, 225],
        'Hobbies': [1, 27, 222, 40, 41, 50, 63, 83, 89, 100, 114],
        'Animals': [3, 32, 54, 64, 113, 127, 192, 153, 204, 216, 223],
        'Reading': [60],
        'LGBTQ+': [5],
        'Sports': [7, 30, 177, 37, 47, 88, 102, 124],
        'Style/Appearance': [8, 10, 18, 35, 42, 92, 120, 125, 171, 195, 227],
        'Education': [9, 52],
        'Geography': [11, 39, 75, 76, 90, 91, 203],
        'Holidays, seasonal, and events': [12, 13, 59, 68, 71, 132, 165, 186, 220],
        'Gaming': [14, 25, 49, 98, 104, 105, 109, 118, 140, 142, 144, 148, 162, 187, 201],
        'Media franchises': [4, 17, 6, 16, 22, 38, 70, 151, 168, 172, 174, 183, 185, 190, 191, 211, 213],
        'Emotional and reflective content': [143, 173],
        'Romantic relationships': [24, 80],
        'Family': [26, 81, 152, 226],
        'Advice': [147],
        'K-pop': [31, 145, 45, 84, 96, 214],
        'Anime and manga': [29, 33, 48, 65, 101, 107, 108, 196, 197, 199, 202, 221, 43, 44, 175, 181, 209, 369],  # Added 369
        'Religion and spirituality': [34, 72, 166, 198],
        'COVID-19': [36, 74],
        'Friendship': [46, 170],
        'Companies/products': [51, 126, 130, 205],
        'Jobs': [57, 110, 122, 164, 200, 218],
        'Playing/producing music': [61, 95, 117, 157, 161, 178, 188],
        'TikTok video types': [67, 69, 86, 103, 134, 207],
        'Mental health': [79, 128, 139, 156, 167, 193],
        'Homemaking': [93, 146],
        'Politics and social issues': [97, 224],
        'Listening to music': [99, 123, 163, 169, 179, 206, 368, 423],  # Added 368 and 423 (Rappers)
        'Celebrities/influencers': [106, 133],
        'Physical health': [138, 189, 217],
        'Incoherent': [2, 19, 20, 21, 23, 28, 53, 55, 56, 58, 62, 77, 78, 82, 85, 94, 111, 115, 119, 121, 135, 136, 137, 141, 149, 150, 155, 158, 159, 160, 176, 180, 182, 184, 208, 212, 215, 219]
    },
    'outlier': {
        'Comedy': [2, 63, 139],
        'Food and drinks': [43, 104],
        'Hobbies': [183, 18, 7, 36, 35, 45, 48, 71, 112, 122, 164, 172, 182],
        'Animals': [8, 144, 162],
        'Reading': [221, 24, 49, 102],
        'Sports': [114, 0, 41, 10, 26, 27, 88, 167, 185, 220],
        'Style/Appearance': [154, 161, 246, 44, 123, 149, 150, 177, 223],
        'Education': [39, 96, 152, 99, 134, 209],
        'Geography': [14, 170, 235, 33, 52, 58, 68, 85, 98, 118, 124, 127, 129, 242],
        'Holidays, seasonal, and events': [60, 11, 106],
        'Gaming': [73, 70, 184, 23, 225, 222, 3, 56, 74, 82, 89, 113, 115],
        'Media franchises': [103, 13, 17, 19, 47, 75, 76, 93, 94, 100, 105, 107, 135, 148, 156, 157, 174, 196, 198, 203, 206, 207, 214, 224, 232, 239, 236],
        'Emotional and reflective content': [15, 69, 117, 120],
        'Romantic relationships': [16, 72, 193, 91, 204, 211],
        'Family': [40],
        'Advice': [6, 92, 25, 50, 57],
        'K-pop': [5, 29, 66, 90, 143, 145, 147],
        'Anime and manga': [22, 37, 67, 131, 179, 230],
        'Friendship': [65, 79],
        'Jobs': [84, 38, 86, 136, 168, 188],
        'Playing/producing music': [97, 189],
        'TikTok video types': [151, 31, 110, 137, 119],
        'Mental health': [219, 51, 62, 101, 155, 200, 228, 231],
        'Politics and social issues': [20, 245],
        'Listening to music': [1, 153, 21, 34, 42, 46, 111, 64, 80, 95, 108, 109, 130, 166, 186, 197],
        'Celebrities/influencers': [12, 59, 81, 116, 133, 180, 234],
        'Physical health': [160, 163, 202, 217],
        'Incoherent': [4, 9, 28, 30, 32, 53, 54, 55, 61, 77, 78, 83, 87, 121, 125, 126, 128, 132, 138, 142, 146, 158, 159, 165, 169, 171, 173, 175, 176, 178, 181, 187, 190, 191, 192, 194, 199, 201, 205, 208, 210, 212, 213, 215, 216, 218, 226, 227, 229, 233, 237, 238, 240, 241, 243, 244, 247]
    }
}

In [21]:
# Create parent topic column
merged_topic_info_with_category = add_categories(
    merged_topic_info=merged_topic_info_renamed,
    category_mapping=category_topic_mapping,
    offset=topic_id_mapping['offset'],
    mapping_format='category_to_topics')

✅ Added categories


In [22]:
# Inspect columns for final renaming
merged_topic_info_with_category.columns

Index(['Topic', 'Count', 'Topic Label', 'Representation',
       'Representative_Docs', 'Source', 'Original Topic Label', 'Category'],
      dtype='object')

In [23]:
merged_topic_info_with_category = merged_topic_info_with_category.rename(
    columns={'Count': 'Document Count',
    'Source': 'Source Model',
    'Topic': 'Topic ID'})

In [24]:
merged_topic_info_with_category.sample(2)

,Topic ID,Document Count,Topic Label,Representation,Representative_Docs,Source Model,Original Topic Label,Category
117,51,9702,iPhone,"[iphone, iphonetricks, ios14, apple, iphonehac...","[shortcut iphone, iphone, peelporn iphone]",Main Model,51_iphone_iphonetricks_ios14_apple,Companies/products
376,388,2167,Disability,"[amputee, disability, disabled, wheelchair, am...",[dyspraxicpov emergencydoctor disability dyspr...,Outlier Model,160_amputee_disability_disabled_wheelchair,Physical health


In [25]:
# Save merged topic info
merged_topic_info_with_category.to_excel(MERGED_TOPICS_CATEGORIES_PATH, index=False)

In [27]:
# Create category document counts
category_dist = merged_topic_info_with_category.groupby('Category').agg({
        'Document Count': 'sum',
        'Topic ID': 'count'
    }).reset_index()

In [28]:
topic_dist = merged_topic_info_with_category.groupby('Topic Label').agg({
        'Document Count': 'sum'
    }).reset_index()

In [29]:
# Save category counts
with pd.ExcelWriter(CATEGORY_COUNT_PATH, engine='openpyxl') as writer:
    category_dist.to_excel(writer, sheet_name='counts_category')

In [30]:
# Save topic counts
with pd.ExcelWriter(TOPIC_COUNT_PATH, engine='openpyxl') as writer:
    topic_dist.to_excel(writer, sheet_name='counts_topics')

# Linking topics to videos

The code below was used to attach the generated topic labels during this pipeline back to the original videos. 

For anonymization, this is greyed out as IDs and possible links to the original videos had to be erased.

In [ ]:
# video_ids = hashtag_df['id']
# len(video_ids)

5332704

In [ ]:
def create_merged_video_topic_mapping(docs_bert, merged_topics, merged_topic_info, video_ids):
    """
    Create final video-topic mapping with merged topics including category.
    """
    print("\n" + "="*70)
    print("CREATING VIDEO-TOPIC MAPPING")
    print("="*70)
    
    # Create topic mappings
    topic_label_map = dict(zip(merged_topic_info['Topic ID'], merged_topic_info['Topic Label']))
    topic_source_map = dict(zip(merged_topic_info['Topic ID'], merged_topic_info['Source']))
    topic_category_map = dict(zip(merged_topic_info['Topic ID'], merged_topic_info['Category']))
    
    # Create DataFrame
    df = pd.DataFrame({
        'video_id': video_ids,
        'document': docs_bert,
        'topic_id': merged_topics,
        'topic_label': [topic_label_map.get(t, 'Outlier') for t in merged_topics],
        'category': [topic_category_map.get(t, 'Uncategorized') for t in merged_topics],
        'source_model': [topic_source_map.get(t, 'Unknown') for t in merged_topics]
    })
    
    print(f"\n✅ Created mapping for {len(df):,} videos")
    print(f"\nDistribution by source:")
    print(df['source_model'].value_counts())
    print(f"\nDistribution by category:")
    print(df['category'].value_counts())
    
    return df

In [ ]:
# Create mapping (make sure you have video_ids!)
# video_topic_df = create_merged_video_topic_mapping(
#     docs_bert=docs_bert,
#     merged_topics=merged_topics,
#     merged_topic_info=merged_topic_info_with_category,
#     video_ids=video_ids
# )


CREATING VIDEO-TOPIC MAPPING

✅ Created mapping for 5,332,704 videos

Distribution by source:
source_model
Main Model       4360743
Outlier Model     971961
Name: count, dtype: int64

Distribution by category:
category
Incoherent                          603440
Media franchises                    592670
Outlier                             569288
Sports                              377497
Style/Appearance                    327443
Hobbies                             308989
Gaming                              256790
Geography                           208408
Holidays, seasonal, and events      186201
Animals                             175033
Listening to music                  172535
Comedy                              134645
Food and drinks                     131399
K-pop                               117978
Anime and manga                     111350
Education                            99141
Jobs                                 90448
Family                               87124
Romant

In [ ]:
# video_topic_df.head()

In [ ]:
# Save
# video_topic_df.to_pickle(DF_PATH)

________________
# AI disclosure statement

AI tools were used to assist:
- developing, labelling, and debugging code
- formatting Markdown cells

AI tools used:
- [CursorAI (Desktop version)](https://cursor.com/agents)
- [ClaudeAI](https://claude.ai/)
- [ChatGPT](https://chatgpt.com/)

I acknowledge my responsibility as a researcher to thoroughly verify all outputs and content produced by AI tools and accept full accountability for their accuracy and validity.

Inga Vondenhof